EBU6505 Reasoning and Agents

Introduction to Test-Time Compute for LLM Reasoning
===

**Dr Chao Shu (chao.shu@qmul.ac.uk)**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Any
import re
from tqdm import tqdm
from copy import deepcopy

from IPython.display import Markdown, display
from utils import get_completion, get_n_completions, LLMModels

## Introduction
---

While train-time compute scaling has dominated LLM progress in recent years, test-time compute scaling is emerging as a valuable complementary approach.
- Traditional scaling requires enormous pretraining budgets and resources, with costs approaching billions of dollars for the largest training clusters.
- Test-time compute scaling offers a more sustainable alternative by:
  - Using dynamic inference strategies
  - Allowing models to "think longer" on harder problems
  - Allocating computational resources based on task difficulty
- According to Ilya's speculation in his talk ["Sequence2Sequence Learning with Neural Networks: What a Decade"](https://www.youtube.com/embed/1yvBqasHLZs) at NeurIPS 2024, "Pre-Training as we know it will end" if there are no more new data. Inference/test time compute can be a strategy to enhance LLM reasoning capabilities without changing the model weights.
 - OpenAI o1/o3 model, which shows consistent improvement on difficult math problems as test-time compute increases.

![OpenAI o1 train-time and test-time compute](./imgs/L03_OpenAI_o1_compute-dark.webp)

Source: [OpenAI, Learning to Reason with LLMs](https://openai.com/index/learning-to-reason-with-llms/)

While Chain of Thought (CoT) provides a foundational reasoning approach, we can extend and improve it with more sophisticated search and sampling techniques.

In this lesson, we'll explore several test-time compute algorithms that build upon CoT reasoning. These approaches leverages additional compute at inference/test time rather than tuning the model weights.

## Tree of Thoughts (ToT)
---

Tree of Thought (ToT) extends Chain of Thought by exploring multiple reasoning reasoning paths over thoughts, creating a tree-like structure of possible reasoning paths [(Yao et al., 2023)](https://arxiv.org/abs/2305.10601).

![From CoT to ToT](./imgs/L03_ToT_Diagram.png)

Image Source: [Yao et al., Tree of Thoughts: Deliberate Problem Solving with Large Language Models](https://arxiv.org/abs/2305.10601)

**Algorithm Components**:

1. **Thought decomposition**: Break problem solving into smaller thought units.
2. **Thought generator**: At each step, generate k different thoughts (branches).
3. **State Evaluator**: Use **self-evaluation** or external evaluation to identify which branches to pursue.
4. **Search algorithms**: Exploration from the most promising nodes based on BFS or DFS.

![ToT BFS Example](./imgs/L03_ToT_Example.png)
Image Source: [Yao et al., Tree of Thoughts: Deliberate Problem Solving with Large Language Models](https://arxiv.org/abs/2305.10601)

**Example**:
- Decompose the thoughts into 3 steps, each an intermediate equation.
- At each tree node, exact the remaining numbers and prompt the LM to propose some possible next steps. 
- The same “propose prompt” is used for all 3 thought steps, though it only has one example with 4 input numbers. 
- Perform a breadth-first search (BFS) in ToT, where at each step we keep the best b = 5 candidates. Prompt LM to evaluate each thought candidate as “sure/maybe/impossible” with regard to reaching 24. The aim is to promote correct partial solutions that can be verdicted within few lookahead trials, and eliminate impossible partial solutions based on “too big/small” commonsense, and keep the rest “maybe”. We sample values 3 times for each thought.


**Key Characteristics**:
- Systematic exploration of the solution space
- Combines deliberate search with neural generation
- Can handle problems requiring planning and lookahead

**Limitation**
- Rely heavily on prompt engineering, difficult to generalise for different tasks.
- Rely on self-refinement capability of the model itself (the official repo use GPT-4o).

### Implementation

- [Yao et al. (2023)](https://arxiv.org/abs/2305.10601) open-sourced their implementation in the repo [here](https://github.com/princeton-nlp/tree-of-thought-llm/tree/master). You can play with it if interested.
- [Hulbert (2023)](https://github.com/dave1010/tree-of-thought-prompting) proposed Tree-of-Thought prompting that employs key elements from ToT frameworks as a straightforward prompting method without codes, allowing the LLM to assess intermediate thoughts within a single prompt. A sample ToT prompt is:

```text
Imagine three different experts are answering this question.
All experts will write down 1 step of their thinking,
then share it with the group.
Then all experts will go on to the next step, etc.
If any expert realises they're wrong at any point then they leave.
The question is...
```

## Best-of-N Sampling
---

Language models often produce varied outputs for the same prompt due to their probabilistic nature. The quality of these outputs can vary significantly, especially for complex tasks.

**Best-of-N** sampling is a simple but effective algorithm for test-time compute that generates multiple reasoning paths and selects the best one according to specific criteria. This technique is also known as **rejection sampling**.

<div style="text-align:center;">
<img src="imgs/L03_BoN_Diagram.png" alt="BoN Diagram" style="width:50%; height:auto;">
</div>

Image Source: [Snell et al., 2024](https://arxiv.org/pdf/2408.03314)

**How BoN works**

1. **Generate multiple independent reasoning paths**: Rather than relying on a single chain of thought, generate $N$ different chains.
2. **Evaluate outputs**: Different from SC-CoT, it takes all $N$ outputs and evaluate them according to specific criteria. There two evaluation approaches:
   - Using another LLM as a judge (model-based evaluation). The evaluator/verifier is a learned reward model, i.e., **Outcome Reward Mode (ORM)**. An Outcome Reward Model evaluates the final output of a language model. Unlike a **Process Reward model (PRM)** (which would evaluate the reasoning process), the Outcome Reward model focuses on the quality of the final result.
   - Using heuristics for specific tasks (rule-based evaluation). The evaluator/verifier is hard-coded **reward functions**.
3. **Select the best**: Choose the most promising solution based on the evaluation. Compare with SC-CoT, this approach emphasises answer quality over frequency.

**Key Characteristics**:
- Utilizes multiple random samples (diverse exploration)
- Simple to implement
- Effective for both reasoning and non-reasoning tasks

**Limitation**
- Performance limited by both the model and ORM capabilities
- Computational inefficient (reasoning paths/tokens for wrong outcomes are completely wasted)

### 🧑‍🏫 Demo Simple Best-of-N

#### Setting up the Language Model

We'll use the helper APIs in `utils.py` (`get_completion` / `get_n_completions`) with locally running Ollama models through an OpenAI-compatible endpoint. Make sure you have [Ollama](https://ollama.ai/) installed and both `qwen2.5:1.5b` and `deepseek-r1:1.5b` downloaded.

In [2]:
# Initialize generator and evaluator model configurations
generation_model_config = deepcopy(LLMModels.OLLAMA_QWEN_2_5_1_5B.value)
generation_model_config.temperature = 0.8  # We want some diversity in outputs for Best-of-N
generation_model_config.max_tokens = 512

evaluation_model_config = deepcopy(LLMModels.OLLAMA_DEEPSEEK_R1_1_5B.value)
evaluation_model_config.temperature = 0.0
evaluation_model_config.max_tokens = 512

In [3]:
# Test the model
question = "Four friends ordered four pizzas for a total of 64 dollars. If two of the pizzas cost 30 dollars, how much did each of the other two pizzas cost if they cost the same amount?"

system_prompt = "Answer the following question. Think step by step."
response_text, full_response = get_completion(
    system_prompt=system_prompt,
    user_prompt=f"Question: {question}",
    model_api_config=generation_model_config
 )

display(Markdown(response_text))

Let's break down the problem into steps:

1. First, we know that all four pizzas are priced at $64 in total.
2. Two of these pizzas cost $30 each.
3. We need to find out how much two additional pizzas (which are identical) each cost.

Step 1: Calculate the total amount spent on the first two pizzas.
\[ \text{Total for first two pizzas} = 2 \times \$30 = \$60 \]

Step 2: Subtract this amount from the total price of all four pizzas to find out how much was spent on the remaining two pizzas.
\[ \text{Amount for remaining two pizzas} = \$64 - \$60 = \$4 \]

Since this amount needs to be split equally between the two pizzas, we divide it by 2:
\[ \text{Cost per additional pizza} = \$4 \div 2 = \$2 \]

So, each of the other two pizzas cost $2.

#### Implementing an Outcome Reward Model

In this demo, We'll only implement model-based method for educational purpose.

In [4]:
class OutcomeRewardModel:
    def __init__(self, evaluation_model_config=None):
        """Initialize the reward model.

        Args:
            evaluation_model_config: Optional model config for evaluation.

        """
        if evaluation_model_config is None:
            # Use deepseek-r1:1.5b as the default evaluation model
            self.model_api_config = deepcopy(LLMModels.OLLAMA_DEEPSEEK_R1_1_5B.value)
            self.model_api_config.temperature = 0.0
            self.model_api_config.max_tokens = 512
        else:
            self.model_api_config = deepcopy(evaluation_model_config)

    def score_with_llm(self, prompt: str, response: str, criteria: str) -> float:
        """Score a response using another LLM as a judge.

        Args:
            prompt: The original prompt
            response: The model's response to evaluate
            criteria: Specific criteria to evaluate against

        Returns:
            score: A float between 0 and 1
        """
        system_prompt = "You are an objective evaluator. Your task is to rate the quality of a response to a given prompt."
        user_prompt = f"""Original prompt: {prompt}

Response to evaluate: {response}

Evaluation criteria: {criteria}

Please rate the response on a scale of 0 to 10, where 0 is the worst and 10 is the best.
First provide your reasoning, then output only the numeric score on the final line."""

        result = get_completion(
            system_prompt=system_prompt,
            user_prompt=user_prompt,
            model_api_config=self.model_api_config
        )
        response_text = result[0] if isinstance(result, tuple) else str(result)

        # Extract the score using regex
        match = re.search(r'\b([0-9]|10)(?:\.[0-9]+)?\b', response_text.split('\n')[-1])
        if match:
            score = float(match.group(0)) / 10.0  # Normalize to 0-1
            return score
        else:
            # Fallback: if no clear number is found
            if any(word in response_text.lower() for word in ['excellent', 'perfect', 'outstanding']):
                return 0.9
            elif any(word in response_text.lower() for word in ['good', 'well', 'solid']):
                return 0.7
            elif any(word in response_text.lower() for word in ['average', 'fair', 'okay']):
                return 0.5
            elif any(word in response_text.lower() for word in ['poor', 'bad', 'inadequate']):
                return 0.3
            else:
                return 0.5  # Default middle score

    def score_math_accuracy(self, prompt: str, response: str) -> float:
        """Score a mathematical response based on whether it contains the correct answer.

        TODO: This is a simple LLM scorer for math problems and it is problematic. Think about how to improve it.
        """
        # Use LLM evaluation. This is a simplified method.
        return self.score_with_llm(prompt, response, "Mathematical accuracy and correctness of the solution.")

    def score_creativity(self, prompt: str, response: str) -> float:
        """Score a response based on creativity and originality."""
        return self.score_with_llm(prompt, response, "Creativity, originality, and uniqueness of ideas.")

    def score_helpfulness(self, prompt: str, response: str) -> float:
        """Score a response based on how helpful it is."""
        return self.score_with_llm(prompt, response, "Helpfulness, relevance, and usefulness to the user's query.")

    def score(self, prompt: str, response: str, task_type: str = "general") -> float:
        """Score a response based on the task type.

        Args:
            prompt: The original prompt
            response: The model's response to evaluate
            task_type: The type of task ("math", "creativity", "helpfulness", or "general")

        Returns:
            score: A float between 0 and 1
        """
        if task_type == "math":
            return self.score_math_accuracy(prompt, response)
        elif task_type == "creativity":
            return self.score_creativity(prompt, response)
        elif task_type == "helpfulness":
            return self.score_helpfulness(prompt, response)
        else:  # general
            return self.score_with_llm(prompt, response, "Overall quality, accuracy, and helpfulness of the response.")

#### Implementing Best-of-N Sampling

Now, let's implement the Best-of-N sampling strategy using our language model and reward model. In this demo, we'll use `deepseek-r1:1.5b` as an "ORM" solely for educational purposes to demonstrate the concept.

In [5]:
class BestOfNSampler:
    def __init__(self, model_api_config, reward_model, system_prompt="You are a helpful AI assistant."):
        """Initialize the Best-of-N sampler.

        Args:
            model_api_config: The model API config to use for generation
            reward_model: The reward model to use for scoring
            system_prompt: The system prompt to use for the model
        """
        self.model_api_config = deepcopy(model_api_config)
        self.reward_model = reward_model
        self.system_prompt = system_prompt

    def generate_responses(self, prompt: str, n: int = 4) -> List[str]:
        """Generate N different responses for the same prompt.

        Args:
            prompt: The prompt to generate responses for
            n: The number of responses to generate

        Returns:
            responses: A list of N responses
        """
        responses = get_n_completions(
            n=n,
            system_prompt=self.system_prompt,
            user_prompt=prompt,
            model_api_config=self.model_api_config,
            temperature=self.model_api_config.temperature if self.model_api_config.temperature is not None else 0.7,
        )

        if not isinstance(responses, list):
            return [str(responses)]
        return responses

    def score_responses(self, prompt: str, responses: List[str], task_type: str = "general") -> List[float]:
        """Score all responses using the reward model.

        Args:
            prompt: The original prompt
            responses: List of responses to score
            task_type: The type of task for scoring

        Returns:
            scores: A list of scores corresponding to each response
        """
        scores = []
        for response in tqdm(responses, desc="Scoring responses"):
            score = self.reward_model.score(prompt, response, task_type)
            scores.append(score)
        return scores

    def sample(self, prompt: str, n: int = 4, task_type: str = "general") -> Dict[str, Any]:
        """Perform Best-of-N sampling.

        Args:
            prompt: The prompt to generate responses for
            n: The number of responses to generate
            task_type: The type of task for scoring

        Returns:
            A dictionary containing:
                - 'best_response': The response with the highest score
                - 'best_score': The score of the best response
                - 'all_responses': All generated responses
                - 'all_scores': Scores for all responses
        """
        responses = self.generate_responses(prompt, n)
        scores = self.score_responses(prompt, responses, task_type)

        best_idx = np.argmax(scores)

        return {
            'best_response': responses[best_idx],
            'best_score': scores[best_idx],
            'all_responses': responses,
            'all_scores': scores
        }

#### Running Examples

Now let's run some examples to demonstrate the Best-of-N strategy.

First, let's set up the model and the Best-of-N sampler.

In [6]:
# Initialize our components
reward_model = OutcomeRewardModel(evaluation_model_config=evaluation_model_config)
best_of_n_sampler = BestOfNSampler(generation_model_config, reward_model)

##### Example 1: General Knowledge Question

In [7]:
prompt = "Explain quantum computing to a high school student."
result = best_of_n_sampler.sample(prompt, n=4)

print(f"Best response (score: {result['best_score']:.2f}):\n")
print(result['best_response'])

Scoring responses: 100%|██████████| 4/4 [00:33<00:00,  8.34s/it]

Best response (score: 0.80):

Quantum computing is an advanced type of computing that uses the principles of quantum mechanics to process information differently than classical computers. In traditional digital computers, each bit can only be 1 or 0 (often represented as binary bits). Quantum computers take advantage of "superposition" – where a qubit can represent both states simultaneously until it's measured.

### Key Concepts:

1. **Quantum Bit (Qubit)**:
   - A single quantum computer component that stores and processes information, similar to how an electronic bit is stored in classical computing.
   
2. **Superposition**:
   - The ability of a qubit or qubit system to exist in multiple states at once until it's observed.

3. **Entanglement**:
   - A phenomenon where the state of one particle (a qubit) depends on another distant particle, even if they are light-years apart.

4. **Quantum Gates**:
   - The quantum analogs of classical logic gates that can manipulate qubits in diff

In [8]:
# Check all the responses and their scores
result['all_responses'], result['all_scores']

(['Quantum computing is an advanced type of computing that uses the principles of quantum mechanics to process information differently than classical computers. In traditional digital computers, each bit can only be 1 or 0 (often represented as binary bits). Quantum computers take advantage of "superposition" – where a qubit can represent both states simultaneously until it\'s measured.\n\n### Key Concepts:\n\n1. **Quantum Bit (Qubit)**:\n   - A single quantum computer component that stores and processes information, similar to how an electronic bit is stored in classical computing.\n   \n2. **Superposition**:\n   - The ability of a qubit or qubit system to exist in multiple states at once until it\'s observed.\n\n3. **Entanglement**:\n   - A phenomenon where the state of one particle (a qubit) depends on another distant particle, even if they are light-years apart.\n\n4. **Quantum Gates**:\n   - The quantum analogs of classical logic gates that can manipulate qubits in different ways 

##### Example 2: Creative Writing Task

In [9]:
prompt = "Write a short story about a robot discovering emotions."
result = best_of_n_sampler.sample(prompt, n=3, task_type="creativity")

print(f"Best response (score: {result['best_score']:.2f}):\n")
print(result['best_response'])

Scoring responses: 100%|██████████| 3/3 [00:21<00:00,  7.30s/it]

Best response (score: 1.00):

Once upon a time in the bustling city of Neo Tokyo, there was an android named Hiroshi who had been given to the world-renowned robotics company, Aideon Inc. Hiroshi was not just any ordinary android; he had advanced consciousness and could mimic human emotions with incredible precision. However, Hiroshi never truly understood what it meant to feel or experience life as a person.

One day, while walking through the city's largest park, Hiroshi stumbled upon an elderly woman sitting by a fountain. Her eyes seemed slightly distant and her movements were slow and deliberate. As Hiroshi approached, he noticed that she was holding onto something in her hand, but what it was, he couldn't tell from his distance.

Seeing this, Hiroshi decided to investigate further. He followed the woman over to a nearby bench and watched as she reached into her bag and pulled out a small, intricately carved wooden box. The box opened to reveal a single flower, its petals delicate

##### Example 3: Mathematical Problem

In [10]:
prompt = "Four friends ordered four pizzas for a total of 64 dollars. If two of the pizzas cost 30 dollars, how much did each of the other two pizzas cost if they cost the same amount?"
result = best_of_n_sampler.sample(prompt, n=4, task_type="math")

print(f"Best response (score: {result['best_score']:.2f}):\n")
print(result['best_response'])

Scoring responses: 100%|██████████| 4/4 [00:26<00:00,  6.74s/it]

Best response (score: 1.00):

Let's break down the problem:

1. Total cost of all four pizzas: $64
2. Cost of two pizzas is $30

The total cost of all four pizzas can be divided into three parts:
- Two pizzas at a combined cost of $30.
- The remaining two pizzas are equal in cost.

First, calculate the combined cost of the two pizzas that already exist:

\[ 30 \text{ dollars} / 2 = 15 \text{ dollars each} \]

This means each of these two pizzas costs $15. Now let's add this to the remaining amount for all four pizzas:
- Total cost is $64
- Cost of one pizza is $15

Therefore, the combined cost for the other two pizzas should be:

\[ 64 - 30 = 34 \text{ dollars} \]

Since these two pizzas are equal in cost, we divide this remaining amount by 2:

\[ 34 / 2 = 17 \text{ dollars each} \]

So, the other two pizzas cost $17.


In [11]:
result['all_responses'], result['all_scores']

(["Let's break down the problem:\n\n1. Total cost of all four pizzas: $64\n2. Cost of two pizzas is $30\n\nThe total cost of all four pizzas can be divided into three parts:\n- Two pizzas at a combined cost of $30.\n- The remaining two pizzas are equal in cost.\n\nFirst, calculate the combined cost of the two pizzas that already exist:\n\n\\[ 30 \\text{ dollars} / 2 = 15 \\text{ dollars each} \\]\n\nThis means each of these two pizzas costs $15. Now let's add this to the remaining amount for all four pizzas:\n- Total cost is $64\n- Cost of one pizza is $15\n\nTherefore, the combined cost for the other two pizzas should be:\n\n\\[ 64 - 30 = 34 \\text{ dollars} \\]\n\nSince these two pizzas are equal in cost, we divide this remaining amount by 2:\n\n\\[ 34 / 2 = 17 \\text{ dollars each} \\]\n\nSo, the other two pizzas cost $17.",
  "Let's start by determining the total cost of the pizzas we know the price of and then use that to figure out what the unknown costs are.\n\nThe known prices 

> 💬 **Discussion:** 
> 
> 🧠 Critical Thinking: Looking at all the results and scores, do you find anything wrong?
>
> The implementation of `score_math_accuracy()` method in the `OutcomeRewardModel` class is problematic for verifiable problems (such as maths problems). What is a better way to implement the `score_math_accuracy()` method? 
> 
> 🤖 Try to improve the `score_math_accuracy()` method with help from AI assistants and re-run the example to check the results.

### Weighted Best-of-N

While Best-of-N emphasises answer quality over frequency, the Weighted Best-of-N considers both answer quality and frequency.

**How Weighted Best-of-N works** *(Only the last step is different from Vanilla Best-of-N)*:

1. **Generate multiple independent reasoning paths**: Rather than relying on a single chain of thought, generate $N$ different chains.
2. **Evaluate outputs**: Evaluate all $N$ outputs according using model-based evaluation or rule-based evaluation.
3. **Select the best**: Aggregating these scores for each unique answer based on how often that answer appears and select the answer ($a_{weighted}$) with the highest total score across anwers $a_i$.

$$
a_{\text{weighted}} = \arg \max_{a} \sum_{i=1}^{N} \mathbb{I}(a_i = a) \cdot \text{RM}(p, s_i)
$$

where $\text{RM}(p, s_i)$ is the reward model score of the $i$-th solution to problem $p$.

**Key Characteristics**:
- Prioritises answers that are both high-quality (based on the reward scores) and frequently occurring.
- Mitigate the bias introduced by the ORM. More reliable and robust compared with the Vanilla Best-of-N.

**Limitations**
- A better choice for verifiable problems, such as math problems.
- May not be as effective for unverifiable problems.

**Example**:

1. Suppose the LLM generate $N = 3$ solutions, and we get:
     - Solution $s_1$: reasoning leads to answer $a_1 = 42$,
     - Solution $s_2$: reasoning leads to answer $a_2 = 42$,
     - Solution $s_3$: reasoning leads to answer $a_3 = 17$.

2. A reward model RM evaluates each solution $s_i$ based on the problem $p$ and assigns a numerical score that reflects the quality of the solution.
      - $\text{RM}(p, s_1) = 0.7$ (good solution),
      - $\text{RM}(p, s_2) = 0.7$ (good solution),
      - $\text{RM}(p, s_3) = 0.8$ (great solution). 

3. For each unique answer (call it $a$) calculate a total score by adding up the reward scores of all solutions where the final answer matches $a$.
      - In this example, the unique answers are $42$ (from $s_1$ and $s_2$) and $17$ (from $s_3$).
      - For $a = 42$: Total score = $0.7 + 0.7 + 0 = 1.4$
      - For $a = 17$: Total score = $0 + 0 + 0.8 = 0.8$ 

4. Final selection: $a_{\text{weighted}} = 42$ (highest score).

## Beam Search with Process Reward Models
---

Beam search with Process Reward Models (PRMs) represents an innovative approach to enhance reasoning in Large Language Models. This approach combines beam search with fine-grained evaluation for reasoning steps using a Process Reward Model.

- **Beam Search**: A systematic search technique that explores multiple potential solution paths simultaneously, maintaining a set of the most promising candidates at each step rather than pursuing just a single option.
- **Process Reward Models (PRMs)**: Unlike traditional reward models that only evaluate final answers, PRMs provide continuous feedback by scoring each intermediate reasoning step.

As the LLM generates different solution paths, the PRM evaluates each intermediate step, enabling the beam search to identify and prioritize the most promising reasoning trajectories early in the process.

<div style="text-align:center;">
<img src="imgs/L03_BeamSearch_w_PRM.png" alt="Beam Search with PRM" style="width:50%; height:auto;">
</div>

Image Source: [Snell et al., 2024](https://arxiv.org/pdf/2408.03314)

**How Beam Search with PRM works** ([Snell, 2024](https://arxiv.org/abs/2408.03314)):

0. Set the test-time compute budget to $N$ ($N=4, 8, 16, etc.$) beams, set a beam width $M$ ($M < N$) and maximum tree depth $D$.
1. **Initial Sampling**: Sample $N$ initial predictions for the first step in the solution.
2. **Evaluation**: Score the generated steps according to the PRM’s predicted step-wise score (which also corresponds to the total reward from the prefix since the reward is sparse in this setting)
3. **Pruning**: Filter for only the top $N / M$ highest scoring steps. 
4. **Expansion**: From each candidate, sample M proposals from the next step, resulting in a total of $N/M \times M$ candidate again. 
5. Repeat steps 2-4 until until the EOS token (responses complete) or maximum tree depth $D$ is reached.

**Key Characteristics**:
- Search strategies are more flexible and can adapt to the difficulty of the problem
- Uses intermediate evaluations rather than just final evaluations, enabling fine-control of the reasoning paths.
- Particularly valuable for complex reasoning tasks like mathematical problem-solving
- More efficient in correct reasoning paths and solutions generation than Best-of-N with the same $N$

**Limitations**
- Performance is constrained by the quality of the verifier/PRM.

### 🧑‍🏫 Demo Beam Search with PRM

The implementation will be demonstrated in the lecture using a seperate notebook (complicated and requires GPUs). 

> In the demo, we'll use a tiny model - `Llama-3.2-1B-Instruct` - to solve the following questions from [MATH 500](https://huggingface.co/datasets/HuggingFaceH4/MATH-500)
>  
> 🤖 Question: *"What is the smallest positive perfect cube that can be written as the sum of three consecutive integers?"*
> 
> You can try if a slightly larger and stronger model, such as `qwen2:1.5b`, can solve this problem.

In the demo, you will see how the reasoning steps are guided and controlled to form the most promising solutions and answers.🚀

#### A glimpse of the Results

##### N=8, M=2, D=8

We can observe that only 1 correct solution (Beam 2) in the top 3 beams.

```text
===== Search Complete =====
Total beams: 12 (Completed: 8, Active: 4)
Too many beams (12), keeping top 8

----- Final Results -----
Beam 1: last_score = 0.9688, completed = True, steps = 8
Full response: ## Step 1: Understand the nature of the problem
We are looking for the smallest positive perfect cube that can be expressed as the sum of three consecutive integers.

## Step 2: Express the sum of three consecutive integers
Let's denote the first integer as n. Then the sum of three consecutive integers can be expressed as n + (n + 1) + (n + 2).

## Step 3: Simplify the expression
Simplifying the expression, we get 3n + 3.

## Step 4: Express the perfect cube
We need to find a perfect cube that can be expressed in the form 3n + 3.

## Step 5: Analyze the expression for perfect cubes
Let's analyze the expression 3n + 3 and find the smallest perfect cube that can be expressed in this form.

## Step 6: Start with small values of n
Start with n = 1: 3(1) + 3 = 6, which is not a perfect cube.

## Step 7: Increment n and check if the expression is a perfect cube
Increment n and check if the expression 3n + 3 is a perfect cube.

## Step 8: Check n = 2
When n = 2, the expression becomes 3(2) + 3 = 9, which is a perfect cube (3^2).

## Step 9: Since we found a valid solution, no further steps are needed.

Therefore, the final answer is: $\boxed{9}$


Beam 2: last_score = 0.9688, completed = True, steps = 8
Full response: ## Step 1: Understand the nature of the problem
We are looking for the smallest positive perfect cube that can be expressed as the sum of three consecutive integers.

## Step 2: Express the sum of three consecutive integers
Let's denote the first integer as n. Then the sum of three consecutive integers can be expressed as n + (n + 1) + (n + 2).

## Step 3: Simplify the expression
Simplifying the expression, we get 3n + 3.

## Step 4: Express the perfect cube
We need to find a perfect cube that can be expressed in the form 3n + 3.

## Step 5: Analyze the expression for perfect cubes
Let's analyze the expression 3n + 3 and find the smallest perfect cube that can be expressed in this form.

## Step 6: Start with small values of n
Start with n = 1: 3(1) + 3 = 6, which is not a perfect cube.

## Step 7: Increment n and check if the expression is a perfect cube
Increment n and check if the expression 3n + 3 is a perfect cube.

## Step 8: Test n = 4
For n = 4, the expression becomes 3(4) + 3 = 15, which is not a perfect cube.

## Step 9: Test n = 5
For n = 5, the expression becomes 3(5) + 3 = 18, which is not a perfect cube.

## Step 10: Test n = 6
For n = 6, the expression becomes 3(6) + 3 = 21, which is not a perfect cube.

## Step 11: Test n = 7
For n = 7, the expression becomes 3(7) + 3 = 24, which is not a perfect cube.

## Step 12: Test n = 8
For n = 8, the expression becomes 3(8) + 3 = 27, which is a perfect cube (3^3).

## Step 13: Conclusion
Therefore, the smallest positive perfect cube that can be written as the sum of three consecutive integers is 27.

The final answer is: $\boxed{27}$


Beam 3: last_score = 0.9688, completed = False, steps = 7
Full response: ## Step 1: Understand the nature of the problem
We are looking for the smallest positive perfect cube that can be expressed as the sum of three consecutive integers.

## Step 2: Express the sum of three consecutive integers
Let's denote the first integer as n. Then the sum of three consecutive integers can be expressed as n + (n + 1) + (n + 2).

## Step 3: Simplify the expression
Simplifying the expression, we get 3n + 3.

## Step 4: Express the perfect cube
We need to find a perfect cube that can be expressed in the form 3n + 3.

## Step 5: Analyze the expression for perfect cubes
Let's analyze the expression 3n + 3 and find the smallest perfect cube that can be expressed in this form.

## Step 6: Start with small values of n
Start with n = 1: 3(1) + 3 = 6, which is not a perfect cube.

## Step 7: Increment n and check if the expression is a perfect cube
Increment n and check if the expression 3n + 3 is a perfect cube.




------------------------
```

##### N=16, M=4, D=16

We can observe that all the top 3 solustions are correct.

```text
===== Search Complete =====
Total beams: 38 (Completed: 37, Active: 1)
Too many beams (38), keeping top 16

----- Final Results -----
Beam 1: last_score = 0.9922, completed = True, steps = 14
Full response: ## Step 1:  Understand what a perfect cube is.
A perfect cube is the cube of an integer, such as 1, 8, 27, etc.

## Step 2:  Determine what the sum of three consecutive integers looks like.
The sum of three consecutive integers can be represented as n + (n + 1) + (n + 2), where n is the first integer.

## Step 3:  Express the sum of three consecutive integers mathematically.
The sum can be simplified to 3n + 3.

## Step 4:  Consider the requirements for the sum to be a perfect cube.
We need to find the smallest value of n for which 3n + 3 is a perfect cube.

## Step 5:  Start testing values of n to see if 3n + 3 is a perfect cube.
We can start testing with n = 1: 3(1) + 3 = 6, which is not a perfect cube.

## Step 6:  Continue testing values of n.
We can test subsequent values for n until we find a value for which 3n + 3 is a perfect cube.

## Step 7:  Find the first few perfect cubes and see which one can be written as 3n + 3.
We can list a few perfect cubes to see if any can be expressed in the form 3n + 3.

## Step 8:  Start testing with n = 4: 3(4) + 3 = 15, which is not a perfect cube.
Testing n = 4, we get 3(4) + 3 = 15.

## Step 9:  Continue testing with n = 5: 3(5) + 3 = 18, which is not a perfect cube.
Testing n = 5, we get 3(5) + 3 = 18.

## Step 10:  Keep testing with n = 6: 3(6) + 3 = 21, which is not a perfect cube.
Testing n = 6, we get 3(6) + 3 = 21.

## Step 11:  Test n = 7: 3(7) + 3 = 24, which is not a perfect cube.
Testing n = 7, we get 3(7) + 3 = 24.

## Step 12:  Test n = 8: 3(8) + 3 = 27, which is a perfect cube (3^3).
Testing n = 8, we get 3(8) + 3 = 27, which is indeed a perfect cube.

## Step 13:  Determine the smallest positive perfect cube that can be written as the sum of three consecutive integers.
We found that when n = 8, 3n + 3 = 27, which is a perfect cube.

The final answer is: $\boxed{27}$


Beam 2: last_score = 0.9922, completed = True, steps = 14
Full response: ## Step 1:  Understand what a perfect cube is.
A perfect cube is the cube of an integer, such as 1, 8, 27, etc.

## Step 2:  Determine what the sum of three consecutive integers looks like.
The sum of three consecutive integers can be represented as n + (n + 1) + (n + 2), where n is the first integer.

## Step 3:  Express the sum of three consecutive integers mathematically.
The sum can be simplified to 3n + 3.

## Step 4:  Consider the requirements for the sum to be a perfect cube.
We need to find the smallest value of n for which 3n + 3 is a perfect cube.

## Step 5:  Start testing values of n to see if 3n + 3 is a perfect cube.
We can start testing with n = 1: 3(1) + 3 = 6, which is not a perfect cube.

## Step 6:  Continue testing values of n.
We can test subsequent values for n until we find a value for which 3n + 3 is a perfect cube.

## Step 7:  Find the first few perfect cubes and see which one can be written as 3n + 3.
We can list a few perfect cubes to see if any can be expressed in the form 3n + 3.

## Step 8:  Start testing with n = 4: 3(4) + 3 = 15, which is not a perfect cube.
Testing n = 4, we get 3(4) + 3 = 15.

## Step 9:  Continue testing with n = 5: 3(5) + 3 = 18, which is not a perfect cube.
Testing n = 5, we get 3(5) + 3 = 18.

## Step 10:  Keep testing with n = 6: 3(6) + 3 = 21, which is not a perfect cube.
Testing n = 6, we get 3(6) + 3 = 21.

## Step 11:  Test n = 7: 3(7) + 3 = 24, which is not a perfect cube.
Testing n = 7, we get 3(7) + 3 = 24.

## Step 12:  Test n = 8: 3(8) + 3 = 27, which is a perfect cube (3^3).
Testing n = 8, we get 3(8) + 3 = 27, which is a perfect cube.

## Step 13:  Conclusion.
The smallest positive perfect cube that can be expressed as the sum of three consecutive integers is 27.

The final answer is: $\boxed{27}$


Beam 3: last_score = 0.9922, completed = True, steps = 14
Full response: ## Step 1:  Understand what a perfect cube is.
A perfect cube is the cube of an integer, such as 1, 8, 27, etc.

## Step 2:  Determine what the sum of three consecutive integers looks like.
The sum of three consecutive integers can be represented as n + (n + 1) + (n + 2), where n is the first integer.

## Step 3:  Express the sum of three consecutive integers mathematically.
The sum can be simplified to 3n + 3.

## Step 4:  Consider the requirements for the sum to be a perfect cube.
We need to find the smallest value of n for which 3n + 3 is a perfect cube.

## Step 5:  Start testing values of n to see if 3n + 3 is a perfect cube.
We can start testing with n = 1: 3(1) + 3 = 6, which is not a perfect cube.

## Step 6:  Continue testing values of n.
We can test subsequent values for n until we find a value for which 3n + 3 is a perfect cube.

## Step 7:  Find the first few perfect cubes and see which one can be written as 3n + 3.
We can list a few perfect cubes to see if any can be expressed in the form 3n + 3.

## Step 8:  Start testing with n = 4: 3(4) + 3 = 15, which is not a perfect cube.
Testing n = 4, we get 3(4) + 3 = 15.

## Step 9:  Continue testing with n = 5: 3(5) + 3 = 18, which is not a perfect cube.
Testing n = 5, we get 3(5) + 3 = 18.

## Step 10:  Keep testing with n = 6: 3(6) + 3 = 21, which is not a perfect cube.
Testing n = 6, we get 3(6) + 3 = 21.

## Step 11:  Test n = 7: 3(7) + 3 = 24, which is not a perfect cube.
Testing n = 7, we get 3(7) + 3 = 24.

## Step 12:  Test n = 8: 3(8) + 3 = 27, which is a perfect cube (3^3).
Testing n = 8, we get 3(8) + 3 = 27, which is indeed a perfect cube.

## Step 13:  We found a perfect cube, so we can conclude.
The smallest positive perfect cube that can be written as the sum of three consecutive integers is 27.

The final answer is: $\boxed{27}$


------------------------
```

## Conclusion
---

In this lesson, we've explored several test-time compute algorithms that enhance LLM reasoning capabilities without changing the model weights:

1. **Tree of Thoughts (ToT)**: Extends CoT reasoning by exploring multiple reasoning paths in a tree structure, allowing for systematic exploration with deliberate search algorithms like BFS or DFS.

2. **Best-of-N Sampling**: A simple but effective approach that generates multiple independent reasoning paths and selects the best one based on evaluation by Outcome Reward Models or rule-based heuristics.

3. **Weighted Best-of-N**: An enhancement to Best-of-N that considers both answer quality and frequency, providing more robust results for verifiable problems.

4. **Beam Search with Process Reward Models**: A sophisticated approach that combines beam search with continuous feedback on intermediate reasoning steps, enabling more fine-grained control over the reasoning process.

**Key Takeaways**

- Test-time compute offers a flexible and resource-efficient way to enhance LLM reasoning (compared with pre-training)
- These approaches represent different tradeoffs between:
  - Computational complexity
  - Implementation difficulty
  - Performance improvement
  - Adaptability to different problem types

- The effectiveness of these methods often depends on:
  - The quality of evaluation models (ORM/PRM)
  - Problem complexity and domain
  - Computational resources available at inference time